In [ ]:
RUNNING_ON_COLAB = True

def run_inference():
    import json
    import os
    import re
    import sys
    from pathlib import Path
    from typing import Optional

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from tqdm import tqdm

    # ── Configuration ──────────────────────────────────────────────────────────────
    MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
    GPU_ID = "0"

    if RUNNING_ON_COLAB:
        PROJECT_ROOT = Path("151B_SP26_Competition")
        sys.path.insert(0, str(PROJECT_ROOT.resolve()))
        DATA_PATH = "/content/151B_SP26_Competition/data/public.jsonl"
        OUTPUT_PATH = "/content/151B_SP26_Competition/results/final_results.jsonl"
    else:
        DATA_PATH = "data/public.jsonl"
        OUTPUT_PATH = "results/final_results.jsonl"

    MAX_TOKENS = 16384
    EVAL_N = 10 # Set to -1 to evaluate on all examples, or a positive integer to evaluate on the first EVAL_N examples (for quick testing)

    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

    print("done")
    print("MODEL_ID:", MODEL_ID)
    print("DATA_PATH:", DATA_PATH)
    print("OUTPUT_PATH:", OUTPUT_PATH)
    print("MAX_TOKENS:", MAX_TOKENS)
    print("EVAL_N:", EVAL_N)


    # ---- Load data and print statistics ───────────────────────────────────────────────
    data = [json.loads(line) for line in open(DATA_PATH)]

    n_mcq  = sum(bool(d.get("options")) for d in data)
    n_free = sum(not d.get("options") for d in data)

    print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form)")

    eval_data = data[:EVAL_N] if EVAL_N > 0 else data

    eval_mcq  = sum(bool(d.get("options")) for d in eval_data)
    eval_free = sum(not d.get("options") for d in eval_data)

    print(f"Evaluating {len(eval_data)} questions ({eval_mcq} MCQ, {eval_free} free-form)")

    # ----- Prompt construction ───────────────────────────────────────────────────────────────
    SYSTEM_PROMPT_MATH = (
        "You are a precise mathematical reasoner. Solve the following problem rigorously. "
        "After obtaining an answer, independently check it for errors or contradictions. Return only the corrected final solution inside \\boxed{}. "
        "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
        "e.g. \\boxed{3, 7}. "
        "Be concise. "
    )
    SYSTEM_PROMPT_MCQ = (
        "You are a precise mathematical reasoner. "
        "Read the problem and the answer choices below, then select the single best answer. "
        "After obtaining an answer, independently check it for errors or contradictions. "
        "Output ONLY the letter of your final chosen option inside \\boxed{}, e.g. \\boxed{C}. "
        "Be concise. "
    )

    def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
        """Return (system_prompt, user_prompt) for a question."""
        if options:
            labels    = [chr(65 + i) for i in range(len(options))]
            opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
            return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
        return SYSTEM_PROMPT_MATH, question


    # ---- Load model and tokenizer ───────────────────────────────────────────────────────────────
    INFERENCE_MODEL = MODEL_ID

    tokenizer = AutoTokenizer.from_pretrained(INFERENCE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    llm = LLM(
        model=INFERENCE_MODEL,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=False,
        gpu_memory_utilization=0.85,
        max_model_len=32768,
        trust_remote_code=True,
        max_num_seqs=16,
        max_num_batched_tokens=8192,
    )

    sampling_params = SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )

    print("Model loaded.")


    # ----- Generate responses for evaluation set ─────────────────────────────────────────────────
    # Build prompts for first EVAL_N public examples
    prompts = []

    for item in eval_data:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    print(f"Generating responses for {len(prompts)} questions with MAX_TOKENS={MAX_TOKENS}...")

    outputs = llm.generate(prompts, sampling_params=sampling_params)
    responses = [out.outputs[0].text.strip() for out in outputs]

    assert len(responses) == len(eval_data), f"Expected {len(eval_data)} responses, got {len(responses)}"


    # ----- Scoring ───────────────────────────────────────────────────────────────
    def extract_letter(text: str) -> str:
        m = re.search(r"\\boxed\{([A-Za-z])\}", text)
        if m:
            return m.group(1).upper()
        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    def score_mcq(response: str, gold_letter: str) -> bool:
        return extract_letter(response) == gold_letter.strip().upper()

    # Load Judger for free-form scoring
    from judger import Judger
    judger = Judger(strict_extract=False)

    results = []

    print("Scoring...")
    for item, response in tqdm(zip(eval_data, responses), total=len(eval_data), desc="Scoring"):
        is_mcq = bool(item.get("options"))
        gold = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append({
            "id": item.get("id"),
            "is_mcq": is_mcq,
            "gold": gold,
            "response": response,
            "correct": correct,
        })

    print(f"Scoring complete. {len(results)} results.")


    # --- Summary statistics ───────────────────────────────────────────────────────────────
    mcq_res  = [r for r in results if r["is_mcq"]]
    free_res = [r for r in results if not r["is_mcq"]]

    def acc(subset):
        return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d} ({acc(mcq_res):.2f}%)")
    print(f"Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d} ({acc(free_res):.2f}%)")
    print(f"Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d} ({acc(results):.2f}%)")
    print("=" * 50)

    # --- Save detailed results to JSONL ───────────────────────────────────────────────────────────────
    out_path = Path(OUTPUT_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        for r in results:
            record = {
                "id": r["id"],
                "is_mcq": r["is_mcq"],
                "gold": r["gold"],
                "response": r["response"],
                "correct": r["correct"],
            }
            f.write(json.dumps(record) + "\n")

    print(f"Saved {len(results)} records to {out_path}")

    # Convert to CSV
    import pandas as pd
    df = pd.DataFrame(results)
    # only keep id, and response for csv
    df = df[["id", "response"]]
    with open(out_path.with_suffix(".csv"), "w") as f:
        df.to_csv(f, index=False)
        print(f"Also saved results to {out_path.with_suffix('.csv')}")


# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment using Python 3.12 explicitly
# !~/.local/bin/uv venv .venv --seed --python 3.12 --clear

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(usually named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [3]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate
print("done")

done


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/verification-pe.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

print("done")

done


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [6]:
SYSTEM_PROMPT_MATH = (
    "You are a precise mathematical reasoner. Solve the following problem rigorously."
    "After obtaining an answer, independently check it for errors or contradictions. Return only the corrected final solution inside \\boxed{}."
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Be concise."
)

SYSTEM_PROMPT_MCQ = (
    "You are a precise mathematical reasoner."
    "Read the problem and the answer choices below, then select the single best answer. "
    "After obtaining an answer, independently check it for errors or contradictions."
    "Output ONLY the letter of your final chosen option inside \\boxed{}, e.g. \\boxed{C}."
    "Be concise."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return "<instructions>" + SYSTEM_PROMPT_MCQ + "</instructions>", f"<problem>{question}\n\nOptions:\n{opts_text}</problem>"
    return "<instructions>" + SYSTEM_PROMPT_MATH + "</instructions>", f"<problem>{question}</problem>"


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))

    print(f"── {label} system prompt ──")
    print(sys_p, "\n")

    print(f"── {label} user prompt ──")
    print(usr_p[:500], "...\n")

── MCQ system prompt ──
<instructions>You are a precise mathematical reasoner. Choose the single best answer. Output only the option letter inside \boxed{}, e.g. \boxed{C}. Do not include reasoning or extra text.</instructions> 

── MCQ user prompt ──
<problem>$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. $frac{1}{a^2}$</problem> ...

── Free-form system prompt ──
<instructions>You are a precise mathematical reasoner. Solve the problem concisely and verify your result. End with the final answer inside \boxed{}. If there are multiple sub-answers, separate them with commas inside one box, e.g. \boxed{3, 7}.</instructions> 

── Free-form user prompt ──
<problem>Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]</problem> ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)


sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("\n === Model loaded. ===\n")

INFO 05-29 23:32:42 [utils.py:278] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-29 23:32:43 [model.py:617] Resolved architecture: Qwen3ForCausalLM


INFO 05-29 23:32:43 [model.py:1752] Using max model len 16384


INFO 05-29 23:32:43 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=32768.


INFO 05-29 23:32:44 [vllm.py:977] Asynchronous scheduling is enabled.


...

(EngineCore pid=1712) 

    torch._C._cuda_init()


(EngineCore pid=1712) 

RuntimeError: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
# Build prompts for all entries
prompts = []

for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate all responses in one vLLM batched pass
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Sanity check: make sure every data item has one response
assert len(responses) == len(data), f"Expected {len(data)} responses, got {len(responses)}"

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!